# PU Learning
#### Document Classification - Deceptive Opinions

Load the dataset

In [ ]:
import pandas as pd

from utils.download import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents.head()

Drop not needed columns

In [ ]:
documents = documents.drop(columns=["hotel", "source"])

Transform the "deceptive" column to it's binary counterpart

In [ ]:
documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
documents.head()

Deceptive & Positive

In [ ]:
deceptive_positive = documents.loc[(documents["deceptive"] == 1) & (documents["polarity"] == "positive")]
deceptive_positive

Deceptive & Negative

In [ ]:
deceptive_negative = documents.loc[(documents["deceptive"] == 1) & (documents["polarity"] == "negative")]
deceptive_negative

Non-deceptive & Positive

In [ ]:
non_deceptive_positive = documents.loc[(documents["deceptive"] == 0) & (documents["polarity"] == "positive")]
non_deceptive_positive

Non-deceptive & Negative

In [ ]:
non_deceptive_negative = documents.loc[(documents["deceptive"] == 0) & (documents["polarity"] == "negative")]
non_deceptive_negative

Transform the dataset to PU compatible

In [ ]:
from pulearn.pubase import *


# Get sample positive indices
positive_indices = extract_sample(documents["deceptive"], ratio=0.45)
# Initialize new column with "unlabeled" values
documents["pu-label"] = 0
# Mark the selected positive data points
documents.loc[positive_indices, "pu-label"] = 1

documents

Initialize text preprocessing steps

In [ ]:
import nltk, re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('omw-1.4')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def prepare(row):
    filtered = []
    row = row.lower().strip()
    row = re.sub('[^A-Za-z0-9.]+', ' ', row)
    row_parts = row.split()
    for part in row_parts:
        if part not in stop_words:
            filtered.append(lemmatizer.lemmatize(part))
    return " ".join(filtered)

In [ ]:
from sklearn.model_selection import train_test_split

from keras_preprocessing.sequence import pad_sequences
from keras.preprocessing.text import Tokenizer

import tensorflow as tf


X = documents["text"].apply(lambda row: prepare(row))
y_pu = documents["pu-label"]

x_train, x_test, y_train, y_test = train_test_split(X, y_pu, test_size=0.2, random_state=42)

Fit the tokenizer on the documents and extract sequences

In [ ]:
# OOV_TOKEN = "<UNK>"

# tokenizer = Tokenizer(oov_token=OOV_TOKEN)
# tokenizer.fit_on_texts(x_train)

# word_index = tokenizer.word_index

# x_train_seq = tokenizer.texts_to_sequences(x_train)
# x_train_seq = pad_sequences(x_train_seq, padding="post")

# x_test_seq = tokenizer.texts_to_sequences(x_test)
# x_test_seq = pad_sequences(x_test_seq, padding="post")

Uncomment below to download Glove embeddings

In [ ]:
# download_embeddings("glove")

Load word embeddings and create matrix

In [ ]:
# from utils.word_embed import *

# embeddings_index = load_embeddings_index(filepath='../embeddings/glove/glove.6B.100d.txt')
# embeddings_matrix = create_embeddings_matrix(embeddings_index, word_index)

Integrate the embeddings matrix in a tensorflow layer

In [ ]:
# from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional


# embedding_layer = Embedding (
#     input_dim = len(word_index) + 1,
#     output_dim = 100,
#     weights = [embeddings_matrix],
#     input_length = 100,
#     trainable = False
# )

Define model's topology

In [ ]:
# from tensorflow.keras.models import Sequential


# model = Sequential ([
#     embedding_layer,
#     Bidirectional(LSTM(150, return_sequences=True)), 
#     Bidirectional(LSTM(150)),
#     Dense(128, activation='relu'),
#     Dense(1, activation='sigmoid')
# ])

# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

Use Zeugma's embeddings downloader

**Takes a lot of time ...**

In [ ]:
from zeugma.embeddings import EmbeddingTransformer

embed_transformer = EmbeddingTransformer('fasttext')

Use cleanlab's pu-learning implementation to train a PU classifier

In [ ]:
from cleanlab import classification
from sklearn.metrics import f1_score
from sklearn.svm import SVC


x_train = embed_transformer.transform(x_train)
x_test = embed_transformer.transform(x_test)

svc = SVC(probability=True)

pu_clf = classification.CleanLearning(clf=svc, pulearning=1)
pu_clf.fit(x_train, y_train)
preds = pu_clf.predict(x_test)

f1_score(y_test, preds)